In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import scanpy as sc

import SCIRA



In [2]:
# st data
st_data = sc.read_h5ad('example_data.h5ad')
st_data.obsm['spatial'] = st_data.obs[['x','y']].values

In [3]:
# pairs
pairs_df = pd.read_csv('RLpairs.csv')
pairs_df['RL'] = pairs_df.R + '_' + pairs_df.L
RL_list = [
    'CD40_CD40LG',
    'CXCR5_CXCL13',
    'ICOS_ICOSLG',
    'CCR7_CCL19',
    'CCR7_CCL21',
]
pairs_df = pairs_df.query("RL in @RL_list").reset_index()


First, we calculate cell-cell pairwise interactions.

In the results here, -1 marks no receptor expression while 0 marks positive receptor expression with absence of ligand.

In [4]:
E_df = SCIRA.Edf(    
    st_data, 
    exp = False, # for scaling in some cases
    fill = 0, # not distinguishing no receptor expression vs no ligand in this example
    sop = True, # multiply instead of only using ligand level
    autocrine = False, # no self-to-self
    radius = 15,
    pairs_df = pairs_df,
    col='label', # column name for cell types
)

In [5]:
col = 'label'
E_df['Sender'] = st_data.obs.loc[E_df['Cell_idx_S'], col].astype(str).values
E_df['Receiver'] = st_data.obs.loc[E_df['Cell_idx_R'], col].astype(str).values
E_df['SR'] = E_df['Sender'] + ' to ' + E_df['Receiver']

Ls = pairs_df.query("RL in @RL_list")['L'].unique()
E_df[Ls] = st_data.to_df().loc[E_df['Cell_idx_S'], Ls].values


This can be aggregated to yield single-cell level signal strengths.

In [ ]:
RL_df = E_df.groupby('Cell_idx_R')[RL_list].sum()

In [ ]:
RL = 'CXCR5_CXCL13'
_, ax = plt.subplots()
plt.scatter(
    x = st_data.obs.loc[RL_df.index, 'x'],
    y = st_data.obs.loc[RL_df.index, 'y'],
    c = RL_df[RL],
    cmap = 'hot',
    vmin = 0,
    s = 1
)
ax.set_facecolor('k')
ax.invert_yaxis()
ax.set_aspect('equal')
ax.autoscale_view()
ax.set_title(RL)

Or, receiver-centric aggregated activation score.

In [ ]:
RA = SCIRA.RA(E_df, RL_list = pairs_df['RL'])

In [ ]:
import seaborn as sns
sns.heatmap(SCIRA.row_norm(RA.T, mode = 'max').T, square = 1)

This can be furthur combiend with proportions of cells expression the receptor to genereate a dotplot. 

In [ ]:
# or using scanpy for dotplots

RL_data = sc.AnnData(
    X = RL_df,
    obs = E_df.value_counts(['Cell_idx_R', 'Receiver']).reset_index().set_index('Cell_idx_R').sort_index(),
)

dp = sc.pl.DotPlot(
    RL_data, 
    var_names = RL_list, 
    groupby = 'Receiver', 
    expression_cutoff = 0, 
    mean_only_expressed = True, 
    # cmap = 'cet_bmy',
    # title = f'{group_title}, Receptor activation',
    vmax = 4,
    vmin = 0,
).swap_axes()
dp.legend(colorbar_title = '.\n\nAverage ligand\n expression\nin neighbors', size_title = 'Fractions of \ncells expressing\n the receptor')
dp.show()

Lastly, it can be aggregated to cell-type pairs for sender-receiver results.

In [ ]:

sr_df = SCIRA.SR_df(E_df, RL_list = pairs_df['RL'])

In [ ]:
sr_df


In [ ]:
# custom heatmap plot function 
def heatmap_colorlabel(df_heatmap,
                       rownames_left=None, rownames_right=None,
                       colnames_top=None, colnames_bot=None,
                       palette_left=None, palette_right=None,
                       palette_top=None, palette_bot=None,
                       xlabel_top=None, xlabel_bot=None,
                       ylabel_left=None, ylabel_right=None,
                       fig_kw=dict(), heatmap_kw=dict(), **args):
    """
    Creates a heatmap with square cells and custom tick labels.
    A dedicated column is reserved for the colorbar.
    
    For each side (top, bottom, left, right):
      - If a palette dictionary is provided, a separate axis is created immediately
        adjacent to the heatmap to show a colored label bar. Tick labels and an axis
        label (set via set_xlabel or set_ylabel) are added to that axis. In this case,
        the corresponding tick labels on the main heatmap axis are removed.
      - Otherwise, the main heatmap axis (or an added twiny axis for the top) is used
        to display tick labels and the axis title.
    
    Parameters:
      df_heatmap      : pandas DataFrame with heatmap data.
      
      rownames_left   : list-like of row labels for the left.
      rownames_right  : list-like of row labels for the right.
      colnames_top    : list-like of column labels for the top.
      colnames_bot    : list-like of column labels for the bottom.
      
      palette_left    : dict mapping each left label to a color.
      palette_right   : dict mapping each right label to a color.
      palette_top     : dict mapping each top label to a color.
      palette_bot     : dict mapping each bottom label to a color.
      
      xlabel_top      : string for the top x-axis label.
      xlabel_bot      : string for the bottom x-axis label.
      ylabel_left     : string for the left y-axis label.
      ylabel_right    : string for the right y-axis label.
      
      fig_kw          : dict of keyword arguments for plt.figure.
      heatmap_kw      : dict of keyword arguments for sns.heatmap.
      **args          : additional keyword arguments for sns.heatmap.
      
    Returns:
      fig             : the matplotlib Figure object.
    """
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    import seaborn as sns
    import matplotlib.patches as patches

    # Helper: group consecutive identical labels.
    def group_consecutive(labels):
        groups = []
        if not labels:
            return groups
        start = 0
        current = labels[0]
        for i, lab in enumerate(labels[1:], start=1):
            if lab != current:
                groups.append((current, start, i))
                current = lab
                start = i
        groups.append((current, start, len(labels)))
        return groups

    # Dimensions of the heatmap.
    n_rows, n_cols = df_heatmap.shape

    # Flags for palette existence.
    has_top_palette = palette_top is not None
    has_bottom_palette = palette_bot is not None
    has_left_palette = palette_left is not None
    has_right_palette = palette_right is not None

    # Flags for label lists.
    has_top_labels = colnames_top is not None
    has_bottom_labels = colnames_bot is not None
    has_left_labels = rownames_left is not None
    has_right_labels = rownames_right is not None

    # Define GridSpec dimensions.
    # Rows: optionally a top palette row + main heatmap row + optionally a bottom palette row.
    nrows_grid = (1 if has_top_palette else 0) + 1 + (1 if has_bottom_palette else 0)
    # Columns: optionally a left palette column + main heatmap column + a colorbar column (always) +
    #          optionally a right palette column.
    ncols_grid = (1 if has_left_palette else 0) + 1 + (1 if has_right_palette else 0)

    # Relative sizes.
    top_ratio = 0.1 if has_top_palette else None
    bottom_ratio = 0.1 if has_bottom_palette else None
    main_row_ratio = 1
    row_heights = []
    if has_top_palette:
        row_heights.append(top_ratio)
    row_heights.append(main_row_ratio)
    if has_bottom_palette:
        row_heights.append(bottom_ratio)

    left_ratio = 1/df_heatmap.shape[1] if has_left_palette else None
    main_col_ratio = 1
    # colorbar_ratio = 0.01  # thinner colorbar
    right_ratio = 1/df_heatmap.shape[1] if has_right_palette else None
    col_widths = []
    if has_left_palette:
        col_widths.append(left_ratio)
    # col_widths.extend([main_col_ratio, colorbar_ratio])
    col_widths.append(main_col_ratio)
    if has_right_palette:
        col_widths.append(right_ratio)

    # Create figure and GridSpec.
    fig = plt.figure(**fig_kw)
    gs = gridspec.GridSpec(nrows=nrows_grid, ncols=ncols_grid,
                           height_ratios=row_heights, width_ratios=col_widths,
                          hspace = 0.01)
    # Allow a small gap only for the colorbar.
    gs.update(wspace=0.1, hspace=0.0)

    # Determine grid indices for main heatmap.
    main_row_idx = 1 if has_top_palette else 0
    main_col_idx = 1 if has_left_palette else 0

    # Create main heatmap axis.
    main_ax = fig.add_subplot(gs[main_row_idx, main_col_idx])
    if 'square' not in heatmap_kw:
        heatmap_kw['square'] = True
    hm = sns.heatmap(df_heatmap, ax=main_ax, cbar=False, **heatmap_kw, **args)
    main_ax.set_xlim(0, n_cols)
    main_ax.set_ylim(n_rows, 0)
    main_ax.set_aspect('equal', adjustable='box')

    # Remove x and y ticks from main_ax if a palette axis will be used on that side.
    if has_top_palette or has_bottom_palette:
        main_ax.set_xticks([])
        main_ax.set_xticklabels([])
    if has_left_palette or has_right_palette:
        main_ax.set_yticks([])
        main_ax.set_yticklabels([])

    # # Create dedicated colorbar axis (always present).
    # cbar_ax = fig.add_subplot(gs[main_row_idx, main_col_idx + 1])
    # cbar = fig.colorbar(hm.collections[0], cax=cbar_ax)
    # # Optionally adjust the colorbar position for spacing.
    # # (Labels on the colorbar are managed by the colorbar itself.)

    # --- Top Side ---
    if has_top_palette:
        for ax_i in range(len(col_widths)):
            ax = fig.add_subplot(gs[0, ax_i])
            if ax_i == main_col_idx:
                top_ax = ax
            else:
                ax.set_visible(False)
                
        # top_ax = fig.add_subplot(gs[0, main_col_idx], )
        
        top_ax.set_xlim(0, n_cols)
        top_ax.set_ylim(0, 1)
        # Remove frame.
        for spine in top_ax.spines.values():
            spine.set_visible(False)
        top_ax.set_yticks([])
        if has_top_labels:
            groups = group_consecutive(colnames_top)
            tick_positions = [start + (end - start) / 2 for (lab, start, end) in groups]
            tick_labels = [str(lab) for (lab, start, end) in groups]
            for (lab, start, end) in groups:
                width = end - start
                color = palette_top.get(lab, 'grey')
                rect = patches.Rectangle((start, 0), width, 1, facecolor=color, edgecolor='none')
                top_ax.add_patch(rect)
            top_ax.set_xticks(tick_positions)
            top_ax.set_xticklabels(tick_labels)
            top_ax.set_xticklabels(tick_labels, rotation = 90) # turn this on/off
        else:
            top_ax.set_xticks([])
        top_ax.xaxis.set_label_position('top')
        top_ax.tick_params(axis='x', which='both', top=True, bottom=False,
                           labeltop=True, labelbottom=False, length=0, pad=2)
        if xlabel_top is not None:
            top_ax.set_xlabel(xlabel_top)
        top_ax.set_aspect(0.5, adjustable='box')
    else:
        if has_top_labels:
            # If no top palette axis, add a twiny axis to display top labels.
            top_labels_ax = main_ax.twiny()
            top_labels_ax.xaxis.set_ticks_position('top')
            top_labels_ax.xaxis.set_label_position('top')
            groups = group_consecutive(colnames_top)
            tick_positions = [start + (end - start) / 2 for (lab, start, end) in groups]
            tick_labels = [str(lab) for (lab, start, end) in groups]
            top_labels_ax.set_xlim(main_ax.get_xlim())
            top_labels_ax.set_xticks(tick_positions)
            top_labels_ax.set_xticklabels(tick_labels, rotation = 90)
            if xlabel_top is not None:
                top_labels_ax.set_xlabel(xlabel_top)
            for spine in top_labels_ax.spines.values():
                spine.set_visible(False)

    # --- Bottom Side ---
    if has_bottom_palette:
        # Remove any existing x-tick labels from main_ax to avoid duplication.
        main_ax.set_xticks([])
        main_ax.set_xticklabels([])
        main_ax.set_xlabel('')

        
        for ax_i in range(len(col_widths)):
            ax = fig.add_subplot(gs[nrows_grid - 1, ax_i])
            if ax_i == main_col_idx:
                bottom_ax = ax
            else:
                ax.set_visible(False)
                
        # bottom_ax = fig.add_subplot(gs[nrows_grid - 1, main_col_idx])
        bottom_ax.set_xlim(0, n_cols)
        bottom_ax.set_ylim(0, 1)
        for spine in bottom_ax.spines.values():
            spine.set_visible(False)
        bottom_ax.set_yticks([])
        if has_bottom_labels:
            groups = group_consecutive(colnames_bot)
            tick_positions = [start + (end - start) / 2 for (lab, start, end) in groups]
            tick_labels = [str(lab) for (lab, start, end) in groups]
            for (lab, start, end) in groups:
                width = end - start
                color = palette_bot.get(lab, 'grey')
                rect = patches.Rectangle((start, 0), width, 1, facecolor=color, edgecolor='none')
                bottom_ax.add_patch(rect)
            bottom_ax.set_xticks(tick_positions)
            bottom_ax.set_xticklabels(tick_labels, rotation = 90)
        else:
            bottom_ax.set_xticks([])
        bottom_ax.xaxis.set_label_position('bottom')
        bottom_ax.tick_params(axis='x', which='both', top=False, bottom=True,
                              labeltop=False, labelbottom=True, length=0, pad=2)
        if xlabel_bot is not None:
            bottom_ax.set_xlabel(xlabel_bot)
        bottom_ax.set_aspect(0.5, adjustable='box')
    else:
        if has_bottom_labels:
            main_ax.xaxis.tick_bottom()
            groups = group_consecutive(colnames_bot)
            tick_positions = [start + (end - start) / 2 for (lab, start, end) in groups]
            tick_labels = [str(lab) for (lab, start, end) in groups]
            main_ax.set_xticks(tick_positions)
            main_ax.set_xticklabels(tick_labels)
            if xlabel_bot is not None:
                main_ax.set_xlabel(xlabel_bot)

    # --- Left Side ---
    if has_left_palette:
        left_ax = fig.add_subplot(gs[main_row_idx, 0])
        left_ax.set_xlim(0, 1)
        left_ax.set_ylim(n_rows, 0)
        for spine in left_ax.spines.values():
            spine.set_visible(False)
        left_ax.set_xticks([])
        if has_left_labels:
            groups = group_consecutive(rownames_left)
            tick_positions = [n_rows - (start + (end - start) / 2) for (lab, start, end) in groups]
            tick_labels = [str(lab) for (lab, start, end) in groups]
            for (lab, start, end) in groups:
                height = end - start
                y_bottom = n_rows - end
                color = palette_left.get(lab, 'grey')
                rect = patches.Rectangle((0, y_bottom), 1, height, facecolor=color, edgecolor='none')
                left_ax.add_patch(rect)
            left_ax.set_yticks(tick_positions)
            left_ax.set_yticklabels(tick_labels)
        else:
            left_ax.set_yticks([])
        left_ax.yaxis.set_label_position('left')
        left_ax.tick_params(axis='y', which='both', left=True, right=False,
                            labelleft=True, labelright=False, length=0, pad=2)
        if ylabel_left is not None:
            left_ax.set_ylabel(ylabel_left, labelpad=10)
            left_ax.yaxis.set_label_coords(-0.35, 0.5)
    else:
        if has_left_labels:
            main_ax.yaxis.tick_left()
            groups = group_consecutive(rownames_left)
            tick_positions = [n_rows - (start + (end - start) / 2) for (lab, start, end) in groups]
            tick_labels = [str(lab) for (lab, start, end) in groups]
            main_ax.set_yticks(tick_positions)
            main_ax.set_yticklabels(tick_labels)
            if ylabel_left is not None:
                main_ax.set_ylabel(ylabel_left)

    # --- Right Side ---
    if has_right_palette:
        right_ax = fig.add_subplot(gs[main_row_idx, -1])
        right_ax.set_xlim(0, 1)
        right_ax.set_ylim(n_rows, 0)
        for spine in right_ax.spines.values():
            spine.set_visible(False)
        right_ax.set_xticks([])
        if has_right_labels:
            groups = group_consecutive(rownames_right)
            tick_positions = [n_rows - (start + (end - start) / 2) for (lab, start, end) in groups]
            tick_labels = [str(lab) for (lab, start, end) in groups]
            for (lab, start, end) in groups:
                height = end - start
                y_bottom = n_rows - end
                color = palette_right.get(lab, 'grey')
                rect = patches.Rectangle((0, y_bottom), 1, height, facecolor=color, edgecolor='none')
                right_ax.add_patch(rect)
            right_ax.set_yticks(tick_positions)
            right_ax.set_yticklabels(tick_labels)
        else:
            right_ax.set_yticks([])
        right_ax.yaxis.set_label_position('right')
        right_ax.tick_params(axis='y', which='both', right=True, left=False,
                             labelright=True, labelleft=False, length=0, pad=2)
        if ylabel_right is not None:
            right_ax.set_ylabel(ylabel_right)
    else:
        if has_right_labels:
            main_ax.yaxis.tick_right()
            groups = group_consecutive(rownames_right)
            tick_positions = [n_rows - (start + (end - start) / 2) for (lab, start, end) in groups]
            tick_labels = [str(lab) for (lab, start, end) in groups]
            main_ax.set_yticks(tick_positions)
            main_ax.set_yticklabels(tick_labels)
            if ylabel_right is not None:
                main_ax.set_ylabel(ylabel_right)

    return fig



df_heatmap = pd.DataFrame(np.random.random(size = (5,16)))
colnames_top = [vv  for v in 'ABCD' for vv in [v] * 4]
colnames_bot = [v for v in 'ABCD'*4]
collabel_top = 'col label top'
collabel_bot = 'col label bot'
rownames_left = [v for v in '12345']
rowlabel_left = 'row label'
heatmap_kw = {'cmap' : 'coolwarm', 'vmax': 1, 'vmin': 0}


col_palette = {
    'A': '#FF0000',
    'B': '#00ff00',
    'C': '#4552e3',
    'D': '#8209e1',
}
fig = heatmap_colorlabel(
    df_heatmap = df_heatmap,
    colnames_top = colnames_top,
    colnames_bot = colnames_bot,
    xlabel_top = collabel_top,
    xlabel_bot = collabel_bot,
    rownames_left = rownames_left,
    ylabel_left = rowlabel_left,
    heatmap_kw = heatmap_kw,
    palette_top = col_palette,
    palette_bot = col_palette,
    fig_kw = {'figsize': (5, 2)},
)

In [ ]:
sr_df.reset_index()['Receiver'].unique()

In [ ]:

# (receiver, sender)
pairs = [
    ('B - GC', 'B - non GC'),
    ('B - GC', 'Follicular Dendritic Cells'),
    ('B - GC', 'Follicular Helper T cells'),
    ('B - non GC', 'B - GC'),
    ('B - non GC', 'Follicular Dendritic Cells'),
    ('B - non GC', 'Follicular Helper T cells'),
    ('Follicular Helper T cells', 'B - GC'),
    ('Follicular Helper T cells', 'B - non GC'),
    ('Follicular Helper T cells', 'Follicular Dendritic Cells'),
    ('LAMP3+ DC', 'CCL19/21+ Fibroblasts'),
    ('TCF7+ CD4 T cells', 'CCL19/21+ Fibroblasts'),
    ('TCF7+ CD8 T cells', 'CCL19/21+ Fibroblasts'),
    
]
RLs = RL_list

palette = {
	'B - non GC': '#FF52FF',
	'B - GC': '#F0E301',
	'Fibroblast': '#7436EB',
	'Endothelial': '#A100CA',
	'Follicular Dendritic Cells': '#869FF0',
	'CCL19/21+ Fibroblasts': '#0240D6',
	'LAMP3+ DC': '#35B44E',
	'TCF7+ CD4 T cells': '#FF9000',
	'TCF7+ CD8 T cells': '#FFA3A3',
	'Follicular Helper T cells': '#FF00BF',
}

df_heatmap = SCIRA.row_norm(sr_df.T, mode = 'max').T.loc[pairs, RLs].T
colnames_top = np.array(pairs)[:, 0].tolist()
colnames_bot = np.array(pairs)[:, 1].tolist()
collabel_top = 'Receiver'
collabel_bot = 'Sender'
rownames_left = RLs
rowlabel_left = 'Receptor_Ligand pairs'
heatmap_kw = {'cmap' : 'hot', 
              'vmax': 1.05, 'vmin': 0.05,
             }


fig = heatmap_colorlabel(
    df_heatmap = df_heatmap,
    colnames_top = colnames_top,
    colnames_bot = colnames_bot,
    xlabel_top = 'Signal Receiving Cell Type\n',
    xlabel_bot = '\nSignal Sending Cell Type',
    rownames_left = rownames_left[::-1],
    ylabel_left = rowlabel_left,
    heatmap_kw = heatmap_kw,
    palette_top = palette,
    palette_bot = palette,
    fig_kw = {'figsize': (13, 3.2)},
)

# add some grids
ax = fig.get_children()[1]

[ax.axvline(i, c = 'w', lw = 2) for i in [3, 6, 9, 10, 11]]

[ax.axhline(i, c = 'w', lw = 0.5) for i in [3]]

for ax in fig.get_children()[2:4]:
    [ax.axvline(i, c = 'w', lw = 2) for i in [3, 6, 9, 10, 11]]
    [ax.axvline(i, c = 'w', lw = 0.8) for i in range(df_heatmap.shape[1])]
